<!-- notebook-header -->
# Support Vector Machines e Kernel Trick

**Modulo:** 03 - Machine Learning  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** Margens, vetores de suporte, kernels, gamma, C e classificacao nao-linear.


# Support Vector Machines e Kernel Trick

## Pre-requisitos e Fio Narrativo**Pre-requisitos:** 0.2 (algebra linear), 0.3 (calculo), 3.1 (classificacao)**Tempo estimado:** 10 horas**Objetivo:** Entender SVM geometricamente, o conceito de margem maxima, e como o kernel trick permite classificacao nao-linear sem computar mapeamentos explicitos.**Fio narrativo:** Ate agora vimos modelos que aprendem fronteiras de decisao diretamente (LR, arvores). SVM aborda o problema de outra forma: busca o hiperplano que **maximiza a margem** entre classes. O kernel trick estende isso para fronteiras nao-lineares sem custo computacional proibitivo. Essa elegancia matematica faz SVM ser um dos modelos mais estudados em ML.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

from itertools import product
from copy import deepcopy

# Datasets
from sklearn.datasets import (
    load_iris, load_breast_cancer, load_diabetes, load_digits,
    fetch_california_housing, make_blobs, make_classification,
    make_regression, make_moons, make_circles,
)

# Pre-processing
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    LabelEncoder, OneHotEncoder, PolynomialFeatures,
)

# Model selection
from sklearn.model_selection import (
    train_test_split, cross_val_score, cross_validate,
    GridSearchCV, RandomizedSearchCV, KFold, StratifiedKFold,
    learning_curve, validation_curve,
)

# Linear models
from sklearn.linear_model import (
    LogisticRegression, LinearRegression, Ridge, Lasso, ElasticNet,
)

# Trees and ensembles
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    GradientBoostingClassifier, GradientBoostingRegressor,
    BaggingClassifier, BaggingRegressor,
    AdaBoostClassifier, AdaBoostRegressor,
    VotingClassifier, StackingClassifier,
)

# Other classifiers
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.svm import SVC, SVR, LinearSVC
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

# Clustering
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture

# Dimensionality reduction
from sklearn.decomposition import PCA, KernelPCA, TruncatedSVD, NMF
from sklearn.manifold import TSNE, Isomap

# Metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    mean_squared_error, mean_absolute_error, r2_score,
    silhouette_score, calinski_harabasz_score, davies_bouldin_score,
)

# Pipeline
from sklearn.pipeline import Pipeline, make_pipeline

print('Setup OK - sklearn:', __import__('sklearn').__version__)

# === DATA SETUP (classificacao) ===
cancer = load_breast_cancer()
X_cancer = cancer.data
y_cancer = cancer.target
feature_names = cancer.feature_names

X_train, X_test, y_train, y_test = train_test_split(
    X_cancer, y_cancer, test_size=0.2, random_state=42, stratify=y_cancer
)
_scaler = StandardScaler().fit(X_train)
X_train_scaled = _scaler.transform(X_train)
X_test_scaled = _scaler.transform(X_test)

# Split adicional treino/validacao usado em alguns notebooks
X_train_fit, X_val_scaled, y_train_fit, y_val = train_test_split(
    X_train_scaled, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# Iris para alguns demos
iris = load_iris()
X_iris = iris.data
y_iris = iris.target
X_iris_scaled = StandardScaler().fit_transform(X_iris)


## 1. Separabilidade Linear e Hiperplano Otimo### Analogia / IntuicaoImagine que voce tem bolinhas vermelhas e azuis numa mesa. Voce quer colocar uma regua para separar as cores. Existem infinitas posicoes possiveis, mas SVM escolhe a posicao que **maximiza a distancia** entre a regua e as bolinhas mais proximas de cada lado. Essas bolinhas mais proximas sao os **vetores de suporte** -- elas "sustentam" a posicao da regua.### Definicao FormalDado um dataset $\{(x_i, y_i)\}$ com $y_i \in \{-1, +1\}$, SVM busca o hiperplano $w \cdot x + b = 0$ que maximiza a margem $\frac{2}{\|w\|}$, sujeito a $y_i(w \cdot x_i + b) \geq 1$ para todo $i$.**Formulacao primal (otimizacao):**$$\min_{w,b} \frac{1}{2}\|w\|^2 \quad \text{s.t.} \quad y_i(w \cdot x_i + b) \geq 1$$### Por que em ML?Maximizar a margem melhora a **generalizacao**: um classificador com margem larga e mais robusto a perturbacoes nos dados. A teoria de Vapnik-Chervonenkis mostra que a margem esta diretamente ligada a capacidade de generalizacao -- quanto maior a margem, menor o risco de overfitting.

In [ ]:
# Visualizar hiperplano em 2D

X, y = make_blobs(n_samples=100, centers=2, n_features=2, random_state=42, cluster_std=1.5)
y = 2*y - 1  # Converter para -1 e +1 para SVM

plt.figure(figsize=(10, 6))
plt.scatter(X[y == 1, 0], X[y == 1, 1], c='red', label='Classe +1', s=100, alpha=0.7)
plt.scatter(X[y == -1, 0], X[y == -1, 1], c='blue', label='Classe -1', s=100, alpha=0.7)

# Dois hiperplanos possiveis
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

# Linha 1: separacao inicial
plt.plot([x_min, x_max], [-1.5, 2], 'g--', linewidth=2, label='Hiperplano 1')
# Linha 2: separacao com margem maior
plt.plot([x_min, x_max], [-0.5, 1.5], 'r--', linewidth=2, label='Hiperplano 2 (melhor margem)')

plt.xlim(x_min, x_max)
plt.ylim(y_min, y_max)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.title('Multiplos hiperplanos separadores')
plt.grid(True, alpha=0.3)
plt.show()

print('SVM busca o hiperplano com maior margem (distancia aos pontos mais proximos)')

In [ ]:
# Treinar SVC linear
svc_linear = SVC(kernel='linear', C=1.0)
svc_linear.fit(X, y)

# Visualizar hiperplano e margem
plt.figure(figsize=(10, 6))
plt.scatter(X[y == 1, 0], X[y == 1, 1], c='red', label='Classe +1', s=100, alpha=0.7)
plt.scatter(X[y == -1, 0], X[y == -1, 1], c='blue', label='Classe -1', s=100, alpha=0.7)

# Marcar vetores de suporte
plt.scatter(X[svc_linear.support_, 0], X[svc_linear.support_, 1], 
           s=200, linewidth=1.5, facecolors='none', edgecolors='black', label='Vetores de Suporte')

# Plotar hiperplano e margens
xx = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 100)
if svc_linear.coef_[0][1] != 0:
    yy = -svc_linear.coef_[0][0] / svc_linear.coef_[0][1] * xx - svc_linear.intercept_[0] / svc_linear.coef_[0][1]
    margin = 1 / np.sqrt(np.sum(svc_linear.coef_[0] ** 2))
    yy_margin1 = yy - margin
    yy_margin2 = yy + margin
    
    plt.plot(xx, yy, 'k-', linewidth=2, label='Hiperplano')
    plt.plot(xx, yy_margin1, 'k--', linewidth=1, alpha=0.5)
    plt.plot(xx, yy_margin2, 'k--', linewidth=1, alpha=0.5)

plt.xlim(X[:, 0].min() - 1, X[:, 0].max() + 1)
plt.ylim(X[:, 1].min() - 1, X[:, 1].max() + 1)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title(f'SVM Linear (C=1.0) - {len(svc_linear.support_)} Vetores de Suporte')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### O que observar- Os vetores de suporte (circulos pretos) sao os pontos **mais proximos** do hiperplano- A margem (regiao entre as linhas tracejadas) e maximizada- Remover qualquer ponto que NAO e vetor de suporte nao muda o hiperplano- O numero de vetores de suporte e geralmente pequeno comparado ao total de pontos### O que concluir- SVM depende apenas dos vetores de suporte, nao de todos os dados -- isso o torna **esparso**- A esparsidade e uma vantagem computacional: predizer depende apenas dos SVs- Em dados limpos e bem separados, poucos SVs significam alta confianca na separacao### Conexao com outros notebooks- Em **3.1 Classificacao**, Logistic Regression tambem define uma fronteira linear, mas sem maximizar margem- Em **0.2 Algebra Linear**, o conceito de produto interno e norma e a base da formulacao SVM

## 2. Hard Margin vs Soft Margin (Parametro C)### Analogia / IntuicaoVoltando a analogia da regua: e se algumas bolinhas estao "do lado errado"? Hard margin exige separacao perfeita (impossivel com ruido). Soft margin permite que **algumas bolinhas violem a margem**, pagando uma penalidade C por violacao. C alto = pouca tolerancia a erros. C baixo = mais tolerancia.### Definicao FormalA formulacao soft margin introduz variaveis de folga $\xi_i \geq 0$:$$\min_{w,b,\xi} \frac{1}{2}\|w\|^2 + C \sum_i \xi_i \quad \text{s.t.} \quad y_i(w \cdot x_i + b) \geq 1 - \xi_i$$- $C \to \infty$: hard margin (exige $\xi_i = 0$)- $C \to 0$: margem maxima, ignora erros### Por que em ML?O parametro C e o controle de **bias-variance** em SVM. C pequeno = alto bias, baixa variancia (underfitting). C grande = baixo bias, alta variancia (overfitting). Tunar C via cross-validation e essencial.

In [ ]:
# Comparar diferentes valores de C
C_values = [0.01, 0.1, 1, 10, 100]
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for idx, C in enumerate(C_values):
    svc = SVC(kernel='linear', C=C)
    svc.fit(X, y)
    
    axes[idx].scatter(X[y == 1, 0], X[y == 1, 1], c='red', alpha=0.7)
    axes[idx].scatter(X[y == -1, 0], X[y == -1, 1], c='blue', alpha=0.7)
    axes[idx].scatter(X[svc.support_, 0], X[svc.support_, 1], 
                      s=150, linewidth=1.5, facecolors='none', edgecolors='black')
    
    xx = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 100)
    if svc.coef_[0][1] != 0:
        yy = -svc.coef_[0][0] / svc.coef_[0][1] * xx - svc.intercept_[0] / svc.coef_[0][1]
        axes[idx].plot(xx, yy, 'k-', linewidth=2)
        margin = 1 / np.sqrt(np.sum(svc.coef_[0] ** 2))
        yy_margin1 = yy - margin
        yy_margin2 = yy + margin
        axes[idx].plot(xx, yy_margin1, 'k--', linewidth=1, alpha=0.5)
        axes[idx].plot(xx, yy_margin2, 'k--', linewidth=1, alpha=0.5)
    
    axes[idx].set_title(f'C={C}, SVs={len(svc.support_)}')
    axes[idx].set_xlim(X[:, 0].min() - 1, X[:, 0].max() + 1)
    axes[idx].set_ylim(X[:, 1].min() - 1, X[:, 1].max() + 1)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('C pequeno: margem grande, mais erros')
print('C grande: margem pequena, menos erros')

### O que observar- Com C=0.01, a margem e enorme mas varios pontos estao dentro ou do lado errado- Com C=100, a margem e estreita e quase todos os pontos estao corretos- O numero de vetores de suporte **diminui** conforme C aumenta- C e um hiperparametro -- nao existe valor "correto" universal### O que concluir- C controla o trade-off bias-variancia diretamente- Em dados ruidosos, C pequeno generaliza melhor (margem larga absorve ruido)- Em dados limpos, C grande pode ser seguro (margem estreita mas precisa)- Sempre tunar C com cross-validation, nunca fixar arbitrariamente### Conexao com outros notebooks- Em **3.2 Regressao**, regularizacao (alpha em Ridge/Lasso) faz o mesmo papel que 1/C em SVM- Em **3.3 Arvores**, max_depth controla complexidade; C controla complexidade em SVM

## 3. Kernel Trick: Nao-Linearidade### Analogia / IntuicaoImagine circulos concentricos: bolinhas vermelhas no centro, azuis em volta. Nenhuma reta separa isso. Mas se voce "levanta" o centro (adiciona uma terceira dimensao $z = x_1^2 + x_2^2$), os pontos ficam separaveis por um plano! O kernel trick faz essa "elevacao" **implicitamente**, sem nunca computar as coordenadas no espaco alto.### Definicao FormalUm kernel $K(x_i, x_j) = \phi(x_i) \cdot \phi(x_j)$ computa o produto interno no espaco mapeado sem computar $\phi$ explicitamente. Kernels comuns:| Kernel | Formula | Espaco ||--------|---------|--------|| Linear | $x_i \cdot x_j$ | Finito, original || Polinomial | $(\gamma x_i \cdot x_j + r)^d$ | Finito, $\binom{n+d}{d}$ dimensoes || RBF (Gaussiano) | $\exp(-\gamma\|x_i - x_j\|^2)$ | **Infinito** || Sigmoid | $\tanh(\gamma x_i \cdot x_j + r)$ | Varia |### Por que em ML?O kernel trick e fundamental porque permite usar SVM em problemas nao-lineares **sem explodir a dimensionalidade**. RBF mapeia para espaco infinito-dimensional, mas o custo computacional depende apenas de $n$ (amostras), nao da dimensionalidade do mapeamento. Isso e computacionalmente elegante e teoricamente poderoso.

In [ ]:
# Dados nao-linearmente separaveis
X_moon, y_moon = make_moons(n_samples=200, noise=0.15, random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

kernels = ['linear', 'poly', 'rbf', 'linear', 'poly', 'rbf']
C_vals = [1, 1, 1, 10, 10, 10]

for idx, (kernel, C) in enumerate(zip(kernels, C_vals)):
    ax = axes[idx // 3, idx % 3]
    
    if kernel == 'poly':
        svc = SVC(kernel=kernel, degree=3, C=C)
    else:
        svc = SVC(kernel=kernel, C=C, gamma='scale')
    
    svc.fit(X_moon, y_moon)
    
    # Plot
    ax.scatter(X_moon[y_moon == 0, 0], X_moon[y_moon == 0, 1], c='blue', alpha=0.6, s=50)
    ax.scatter(X_moon[y_moon == 1, 0], X_moon[y_moon == 1, 1], c='red', alpha=0.6, s=50)
    ax.scatter(X_moon[svc.support_, 0], X_moon[svc.support_, 1], 
              s=100, linewidth=1.5, facecolors='none', edgecolors='black')
    
    # Decision boundary
    h = 0.02
    x_min, x_max = X_moon[:, 0].min() - 0.5, X_moon[:, 0].max() + 0.5
    y_min, y_max = X_moon[:, 1].min() - 0.5, X_moon[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = svc.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    ax.contour(xx, yy, Z, levels=[0], linewidths=2, colors='green')
    ax.contourf(xx, yy, Z, levels=np.linspace(Z.min(), Z.max(), 20), cmap='RdBu_r', alpha=0.2)
    
    acc = svc.score(X_moon, y_moon)
    ax.set_title(f'{kernel.upper()} (C={C}, Acc={acc:.3f})')
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### O que observar- Kernel linear falha completamente em dados curvos (moons)- RBF captura a fronteira curva com precisao- Polinomial grau 3 tambem funciona mas com fronteira menos suave- Sigmoid pode ser instavel e raramente e a melhor escolha### O que concluir- A escolha do kernel e uma decisao de modelagem critica- RBF e o kernel padrao e funciona bem na maioria dos casos- Kernel linear e equivalente a LinearSVC (mais rapido para dados grandes)- Nunca assumir que um kernel e melhor sem testar com cross-validation### Conexao com outros notebooks- Em **0.2 Algebra Linear**, produto interno e a base do kernel trick- Em **3.1 Classificacao**, fronteiras de decisao lineares limitam LR; kernels superam isso

## 4. Comparacao Detalhada de Kernels### Por que em ML?Diferentes datasets requerem diferentes kernels. Dados com circulos concentricos precisam de RBF ou polinomial, enquanto dados textuais em alta dimensao funcionam bem com kernel linear. A regra pratica: comece com RBF, teste linear se n_features >> n_samples.

In [ ]:
# Comparar kernels no dataset circles
X_circles, y_circles = make_circles(n_samples=200, noise=0.1, random_state=42)

kernels_compare = ['linear', 'poly', 'rbf', 'sigmoid']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for idx, kernel in enumerate(kernels_compare):
    if kernel == 'poly':
        svc = SVC(kernel=kernel, degree=2, C=10)
    else:
        svc = SVC(kernel=kernel, C=10, gamma='scale')
    
    svc.fit(X_circles, y_circles)
    
    # Decision boundary
    h = 0.02
    x_min, x_max = X_circles[:, 0].min() - 0.5, X_circles[:, 0].max() + 0.5
    y_min, y_max = X_circles[:, 1].min() - 0.5, X_circles[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = svc.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    axes[idx].scatter(X_circles[y_circles == 0, 0], X_circles[y_circles == 0, 1], c='blue', alpha=0.6, s=50)
    axes[idx].scatter(X_circles[y_circles == 1, 0], X_circles[y_circles == 1, 1], c='red', alpha=0.6, s=50)
    axes[idx].contourf(xx, yy, Z, levels=np.linspace(Z.min(), Z.max(), 20), cmap='RdBu_r', alpha=0.2)
    axes[idx].contour(xx, yy, Z, levels=[0], linewidths=2, colors='green')
    
    acc = svc.score(X_circles, y_circles)
    axes[idx].set_title(f'{kernel.upper()} (Acc={acc:.3f})')
    axes[idx].set_xlim(x_min, x_max)
    axes[idx].set_ylim(y_min, y_max)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### O que observar- Em dados com circulos concentricos, linear falha completamente (acc ~50%)- Polinomial grau 2 captura o padrao circular perfeitamente- RBF tambem captura, com fronteira mais suave- Sigmoid tem desempenho inconsistente### O que concluir- A geometria dos dados determina o kernel ideal- Para circulos/esferas: polinomial grau 2 ou RBF- Para meias-luas: RBF ou polinomial grau 3+- Na pratica, RBF e o kernel mais versatil e seguro como ponto de partida### Conexao com outros notebooks- Em **3.3 Arvores**, arvores de decisao fazem splits retangulares; kernels fazem fronteiras suaves- Em **3.6 Reducao de Dimensionalidade**, kernel PCA usa exatamente os mesmos kernels

## 5. RBF Kernel: O Parametro Gamma### Analogia / IntuicaoGamma controla o "raio de influencia" de cada exemplo. Gamma pequeno = cada ponto influencia pontos distantes (visao de longe, padrao suave). Gamma grande = cada ponto so influencia vizinhos proximos (visao de perto, padrao detalhado). E como ajustar o zoom de uma camera: zoom in demais vira ruido, zoom out demais perde detalhes.### Definicao FormalPara RBF: $K(x_i, x_j) = \exp(-\gamma\|x_i - x_j\|^2)$- $\gamma$ pequeno: $K \approx 1$ para muitos pares (influencia ampla)- $\gamma$ grande: $K \approx 0$ para pares distantes (influencia local)- Relacao com largura da Gaussiana: $\gamma = \frac{1}{2\sigma^2}$### Por que em ML?Gamma e tao importante quanto C para o desempenho. Um gamma mal escolhido pode transformar um bom modelo em lixo. A interacao C-gamma cria um espaco de hiperparametros 2D que precisa ser explorado sistematicamente via grid search.

In [ ]:
# Comparar diferentes valores de gamma
gammas = [0.001, 0.01, 0.1, 1, 10]
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for idx, gamma in enumerate(gammas):
    svc = SVC(kernel='rbf', C=10, gamma=gamma)
    svc.fit(X_moon, y_moon)
    
    h = 0.02
    x_min, x_max = X_moon[:, 0].min() - 0.5, X_moon[:, 0].max() + 0.5
    y_min, y_max = X_moon[:, 1].min() - 0.5, X_moon[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = svc.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    axes[idx].scatter(X_moon[y_moon == 0, 0], X_moon[y_moon == 0, 1], c='blue', alpha=0.6, s=50)
    axes[idx].scatter(X_moon[y_moon == 1, 0], X_moon[y_moon == 1, 1], c='red', alpha=0.6, s=50)
    axes[idx].contourf(xx, yy, Z, levels=np.linspace(Z.min(), Z.max(), 20), cmap='RdBu_r', alpha=0.2)
    axes[idx].contour(xx, yy, Z, levels=[0], linewidths=2, colors='green')
    
    acc = svc.score(X_moon, y_moon)
    axes[idx].set_title(f'gamma={gamma} (Acc={acc:.3f})')
    axes[idx].set_xlim(x_min, x_max)
    axes[idx].set_ylim(y_min, y_max)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Gamma pequeno: padrao suave, pode underfittar')
print('Gamma grande: padrao irregular, pode overfittar')

### O que observar- Gamma=0.001: fronteira quase linear (underfitting)- Gamma=0.1-1: fronteira se adapta bem ao formato dos dados- Gamma=10: fronteira excessivamente irregular (overfitting -- ilhas em torno de pontos individuais)- Acuracia no treino com gamma alto pode ser perfeita, mas generalizacao sera pessima### O que concluir- Gamma e o controle de complexidade do kernel RBF- Existe um "sweet spot" que depende da escala dos dados- `gamma='scale'` (padrao sklearn) usa $\gamma = \frac{1}{n_{features} \cdot \text{var}(X)}$ -- bom ponto de partida- Sempre tunar gamma junto com C, nunca separadamente### Conexao com outros notebooks- Em **3.3 Arvores**, max_depth controla overfitting; gamma faz o equivalente em SVM- Em **3.5 Clustering**, DBSCAN tem epsilon que funciona como 1/gamma (raio de vizinhanca)

## 6. Grid Search para SVM### Por que em ML?SVM com RBF tem dois hiperparametros criticos (C e gamma) que interagem fortemente. Um grid search 2D com cross-validation e a forma padrao de tunar SVM. Sem isso, SVM pode parecer que "nao funciona" quando na verdade os hiperparametros estao errados.

In [ ]:
# Grid search no dataset Breast Cancer
data = load_breast_cancer()
X_bc, y_bc = data.data, data.target

X_train_bc, X_test_bc, y_train_bc, y_test_bc = train_test_split(
    X_bc, y_bc, test_size=0.2, random_state=42, stratify=y_bc
)

scaler = StandardScaler()
X_train_bc = scaler.fit_transform(X_train_bc)
X_test_bc = scaler.transform(X_test_bc)

C_range = [0.1, 1, 10, 100]
gamma_range = ['scale', 'auto', 0.001, 0.01, 0.1, 1]

svc_grid = SVC(kernel='rbf', probability=True)
param_grid = {'C': C_range, 'gamma': gamma_range}

grid_search = GridSearchCV(svc_grid, param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1)
grid_search.fit(X_train_bc, y_train_bc)

print('\n=== GRID SEARCH RESULTS ===')
print(f'Melhores parametros: {grid_search.best_params_}')
print(f'Best F1-Score (CV): {grid_search.best_score_:.4f}')

best_svc = grid_search.best_estimator_
acc_best = best_svc.score(X_test_bc, y_test_bc)
print(f'Acuracia no Teste: {acc_best:.4f}')

### O que observar- O grid search testa combinacoes de C e gamma sistematicamente- O melhor par (C, gamma) e escolhido pelo F1-Score medio em 5-fold CV- A importancia de `StandardScaler` antes de SVM: features em escalas diferentes distorcem a distancia- O tempo de treinamento cresce com o tamanho do grid### O que concluir- Normalizar SEMPRE antes de SVM (e qualquer modelo baseado em distancia)- O grid deve cobrir varias ordens de magnitude (ex: C = 0.1 a 100, gamma = 0.001 a 1)- RandomizedSearchCV pode ser mais eficiente para grids grandes- Para datasets grandes (>10k amostras), considerar LinearSVC + GridSearch primeiro### Conexao com outros notebooks- Em **3.1 Classificacao**, GridSearchCV tambem foi usado para tunar outros modelos- Em **3.2 Regressao**, alpha em Ridge/Lasso e o hiperparametro analogo a C

## 7. SVR (Support Vector Regression)### Analogia / IntuicaoNa classificacao, SVM busca uma margem **vazia** entre classes. Na regressao, SVR busca um **tubo** em torno da predicao onde erros sao "tolerados" (nao penalizados). O parametro epsilon define a largura do tubo. Pontos fora do tubo sao penalizados -- esses sao os vetores de suporte da regressao.### Definicao FormalSVR minimiza: $\frac{1}{2}\|w\|^2 + C \sum_i (\xi_i + \xi_i^*)$Sujeito a: $|y_i - (w \cdot x_i + b)| \leq \epsilon + \xi_i$O epsilon-tube ignora erros menores que $\epsilon$, focando apenas nos erros grandes.### Por que em ML?SVR e robusto a outliers (diferente de regressao linear) porque a loss function e epsilon-insensitive: erros pequenos dentro do tubo sao ignorados. Isso o torna util em series temporais ruidosas e problemas de regressao com outliers.

In [ ]:
# SVR em dados sinteticos

X_svr = np.linspace(0, 10, 100).reshape(-1, 1)
y_svr = np.sin(X_svr).ravel() + np.random.normal(0, 0.2, 100)

epsilons = [0.1, 0.5, 1.0]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, eps in enumerate(epsilons):
    svr = SVR(kernel='rbf', C=10, epsilon=eps, gamma='scale')
    svr.fit(X_svr, y_svr)
    
    X_range = np.linspace(0, 10, 300).reshape(-1, 1)
    y_pred = svr.predict(X_range)
    
    axes[idx].scatter(X_svr, y_svr, alpha=0.6, s=50, label='Dados')
    axes[idx].plot(X_range, y_pred, 'r-', linewidth=2, label='Predicao')
    axes[idx].fill_between(X_range.ravel(), y_pred - eps, y_pred + eps, alpha=0.2, color='red', label='epsilon-tube')
    
    axes[idx].set_title(f'epsilon={eps}')
    axes[idx].set_ylabel('y')
    axes[idx].set_xlabel('x')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### O que observar- Com epsilon pequeno (0.1): tubo estreito, predicao acompanha dados de perto- Com epsilon grande (1.0): tubo largo, predicao mais suave e menos sensivel a ruido- Os vetores de suporte sao os pontos FORA do tubo- Epsilon grande = menos SVs = modelo mais simples### O que concluir- Epsilon controla a suavidade da regressao (analogo a regularizacao)- Para dados ruidosos, epsilon maior generaliza melhor- SVR combina com kernel RBF para regressao nao-linear- Na pratica, SVR e menos usado que Random Forest para regressao devido a necessidade de tuning### Conexao com outros notebooks- Em **3.2 Regressao**, comparamos LR, Ridge, Lasso -- SVR e a alternativa kernel-based- Em **3.3 Arvores**, RF Regressor e a alternativa mais comum a SVR na pratica

## 8. Exercicios Praticos### Exercicio 1: Kernel ComparisonCompare todos os 4 kernels (linear, poly, rbf, sigmoid) no dataset Make Moons. Qual funciona melhor e por que?

In [ ]:
# TAREFA DO ALUNO: Exercicio 1 - Comparar kernels em make_moons# Use SVC com cada kernel, compute acuracia, e analise os resultadoskernels_test = None  # TAREFA DO ALUNO: lista de kernelsresults = None  # TAREFA DO ALUNO: dicionario kernel -> acuracia

In [ ]:
# SOLUCAO - Exercicio 1

X_moon, y_moon = make_moons(n_samples=200, noise=0.15, random_state=42)

for kernel in ['linear', 'poly', 'rbf', 'sigmoid']:
    if kernel == 'poly':
        svc = SVC(kernel=kernel, degree=3, C=10)
    else:
        svc = SVC(kernel=kernel, C=10, gamma='scale')
    svc.fit(X_moon, y_moon)
    acc = svc.score(X_moon, y_moon)
    print(f'{kernel}: {acc:.4f}')

print('\nRBF funciona melhor em dados nao-lineares como moons')

### Exercicio 2: Gamma TuningVarie gamma de 0.001 a 100 em escala logaritmica. Plote acuracia treino vs teste. Qual gamma balanceia melhor?

In [ ]:
# TAREFA DO ALUNO: Exercicio 2 - Gamma tuning com validacao# Use np.logspace(-3, 2, 20) para gerar gammas# Treine SVC RBF com cada gamma, compute acuracia treino e testegammas_test = None  # TAREFA DO ALUNO: array de gammastrain_scores = None  # TAREFA DO ALUNO: lista de acuracias treinotest_scores = None  # TAREFA DO ALUNO: lista de acuracias teste

In [ ]:
# SOLUCAO - Exercicio 2

X_m, y_m = make_moons(n_samples=300, noise=0.2, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_m, y_m, test_size=0.3, random_state=42)

gammas_range = np.logspace(-3, 2, 20)
train_accs = []
test_accs = []

for gamma in gammas_range:
    svc = SVC(kernel='rbf', C=10, gamma=gamma)
    svc.fit(X_tr, y_tr)
    train_accs.append(svc.score(X_tr, y_tr))
    test_accs.append(svc.score(X_te, y_te))

plt.figure(figsize=(10, 5))
plt.plot(gammas_range, train_accs, 'o-', label='Treino', linewidth=2)
plt.plot(gammas_range, test_accs, 's-', label='Teste', linewidth=2)
plt.xlabel('Gamma')
plt.ylabel('Acuracia')
plt.xscale('log')
plt.title('Gamma Tuning: Treino vs Teste')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print('Gamma muito alto -> overfitting (treino=1.0, teste cai)')
print('Gamma muito baixo -> underfitting (ambos baixos)')

### Exercicio 3: C vs Vetores de SuporteMostre como C afeta o numero de vetores de suporte e a acuracia. C maior significa mais ou menos SVs?

In [ ]:
# TAREFA DO ALUNO: Exercicio 3 - Relacao C vs SVs# Use C_range = [0.01, 0.1, 1, 10, 100]# Para cada C, conte support vectors e compute acuraciaC_range_ex = None  # TAREFA DO ALUNO: lista de Cnum_svs = None  # TAREFA DO ALUNO: lista de contagens de SVsaccs_ex = None  # TAREFA DO ALUNO: lista de acuracias

In [ ]:
# SOLUCAO - Exercicio 3

X_m, y_m = make_moons(n_samples=200, noise=0.15, random_state=42)
C_range_test = [0.01, 0.1, 1, 10, 100]
svs = []
accs = []

for C in C_range_test:
    svc = SVC(kernel='rbf', C=C, gamma='scale')
    svc.fit(X_m, y_m)
    svs.append(len(svc.support_))
    accs.append(svc.score(X_m, y_m))

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(C_range_test, svs, 'o-', label='Num SV', color='blue', linewidth=2)
ax1.set_xlabel('C')
ax1.set_ylabel('Numero de Vetores de Suporte', color='blue')
ax1.set_xscale('log')
ax1.tick_params(axis='y', labelcolor='blue')

ax2 = ax1.twinx()
ax2.plot(C_range_test, accs, 's-', label='Acuracia', color='red', linewidth=2)
ax2.set_ylabel('Acuracia', color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.title('C: Impact on SV count and Accuracy')
fig.tight_layout()
plt.show()
print('C maior reduz o numero de SVs porque tolera menos erros')

### O que observar nos exercicios- O exercicio 1 mostra que a escolha de kernel depende da geometria dos dados- O exercicio 2 demonstra visualmente o trade-off bias-variancia via gamma- O exercicio 3 conecta C com a esparsidade do modelo (menos SVs = modelo mais simples)- Em todos os exercicios, SVM requer tuning cuidadoso -- nao e um modelo "plug and play"### O que concluir sobre os exercicios- SVM e poderoso mas sensivel a hiperparametros- A combinacao C + gamma + kernel cria um espaco de busca 3D que precisa ser explorado- Grid Search com CV e obrigatorio para SVM funcionar bem### Conexao com outros notebooks- Em **3.1 Classificacao**, outros modelos (LR, arvores) sao menos sensiveis a hiperparametros- Em **3.3 Ensemble**, bagging/boosting ajustam complexidade via n_estimators (mais intuitivo que C/gamma)### Por que em ML?A sensibilidade de SVM a hiperparametros e uma fraqueza pratica: em competicoes e projetos reais, muitos praticantes preferem tree-based models (RF, XGBoost) porque requerem menos tuning. SVM brilha em datasets pequenos e de alta dimensionalidade.

### O que observar no panorama geral- SVM e o unico modelo linear que maximiza margem -- isso tem garantias teoricas de generalizacao- O kernel trick e uma das ideias mais elegantes de ML: transforma problemas nao-lineares em lineares- A dualidade entre C (soft margin) e gamma (complexidade do kernel) controla todo o comportamento- SVR estende a mesma ideia para regressao com o epsilon-tube### O que concluir sobre o panorama geral- SVM e fundamentalmente diferente de arvores e modelos lineares -- entender essa diferenca expande sua caixa de ferramentas- Na pratica moderna, SVM e mais usado com kernel linear em textos (NLP) e com RBF em datasets pequenos- Para datasets grandes (>100k amostras), LinearSVC ou SGDClassifier sao preferidos por escalabilidade### Conexao com outros notebooks- Em **3.5 Clustering**, kernel k-means usa kernels da mesma forma que SVM- Em **3.6 Reducao**, kernel PCA aplica o mesmo truque para componentes principais nao-lineares

### O que observar sobre Escalabilidade

- SVM com kernel RBF e O(n^2) em memoria -- para 100k amostras, a matrix de kernel tem 10 bilhoes de entradas
- LinearSVC usa algoritmo liblinear que e O(n) -- viavel para milhoes de amostras

### O que concluir sobre Escalabilidade

- A escolha entre SVC e LinearSVC depende fundamentalmente do tamanho do dataset
- Em NLP com bag-of-words (milhares de features esparsas), LinearSVC e frequentemente o melhor modelo

### Conexao com outros notebooks

- Em **2.3 SQL/APIs**, dados grandes podem exigir sampling antes de SVM
- Em **3.1 Classificacao**, SGDClassifier com loss='hinge' e SVM linear com SGD

## 9. Erros Comuns e Armadilhas### Erro 1: Nao Normalizar os DadosSVM e baseado em distancia (norma de w, produto interno). Se Feature 1 vai de 0 a 1000 e Feature 2 vai de 0 a 1, Feature 1 domina completamente. **Sempre** usar StandardScaler ou MinMaxScaler antes de SVM.### Erro 2: Gamma Padrao sem Validacao`gamma='scale'` e um default razoavel, mas NAO e otimo. Em muitos datasets, o gamma ideal esta longe do default. Sempre incluir gamma no grid search.### Erro 3: Usar SVM em Datasets Grandes sem AdaptarSVM tem complexidade O(n^2) a O(n^3) em memoria/tempo. Para >50k amostras, usar LinearSVC (que usa liblinear, O(n)) ou SGDClassifier(loss='hinge') que e SVM com SGD.### Erro 4: Esquecer de Tunar CC=1.0 (default) raramente e o melhor valor. A performance de SVM pode variar drasticamente com C. Testar pelo menos C = [0.01, 0.1, 1, 10, 100].### Erro 5: Kernel Polinomial com Grau AltoKernel poly com degree > 5 e numericamente instavel e propenso a overfitting severo. Na pratica, degree=2 ou 3 sao os unicos valores uteis.### Erro 6: Interpretar SVM como Probabilidade`SVC(probability=True)` usa calibracao Platt para estimar probabilidades, mas estas sao **calibradas post-hoc** e podem ser imprecisas. Para probabilidades confiaveis, preferir Logistic Regression ou calibrar explicitamente.### Erro 7: Ignorar a Escala de Features Novas em ProducaoO scaler deve ser **fitted no treino** e **aplicado no teste e producao**. Usar `scaler.transform()` (nao `fit_transform()`) em dados novos.

## 10. Resumo e Conexoes### Hierarquia de Conceitos```SVM|-- Classificacao (SVC)|   |-- Linear (kernel='linear')|   |   |-- Hard Margin (C -> inf)|   |   +-- Soft Margin (C finito)|   +-- Nao-Linear (kernel trick)|       |-- Polinomial (degree)|       |-- RBF (gamma)|       +-- Sigmoid|-- Regressao (SVR)|   |-- Epsilon-tube|   +-- Mesmos kernels+-- Hiperparametros    |-- C (trade-off margem/erro)    |-- gamma (complexidade kernel)    +-- epsilon (largura tubo SVR)```### Conexoes com Outros Notebooks| Conceito | Notebook | Relacao ||----------|----------|---------|| Fronteira linear | 3.1 Classificacao | LR vs SVM linear -- margem vs likelihood || Regularizacao | 3.2 Regressao | C em SVM <==> 1/alpha em Ridge || Overfitting | 3.3 Arvores | max_depth <==> gamma || Kernel | 3.6 Reducao | Kernel PCA usa mesmos kernels || Escalabilidade | 3.5 Clustering | DBSCAN eps <==> 1/gamma || Normalizacao | 2.1 Limpeza | StandardScaler obrigatorio |### Checklist de Competencias- [ ] Explicar geometricamente o que SVM maximiza- [ ] Diferenciar hard margin e soft margin- [ ] Explicar o kernel trick sem formulas- [ ] Escolher kernel adequado para diferentes geometrias de dados- [ ] Tunar C e gamma via grid search com CV- [ ] Saber quando SVM e preferivel a arvores/ensemble- [ ] Usar SVR e entender o epsilon-tube- [ ] Normalizar features antes de SVM### Proximos PassosNo notebook **3.5 Clustering**, veremos algoritmos nao-supervisionados que agrupam dados sem labels. Interessantemente, kernel k-means usa o mesmo truque de kernel que SVM -- mostrando como uma ideia poderosa transcende classificacao e regressao.